# Ensemble Learning

## Objetivo

Compreender como a combinação de múltiplos modelos pode melhorar o desempenho preditivo e a robustez das previsões.

Ao longo deste estudo, serão revisados os fundamentos de Ensemble Learning, passando por Bagging e Boosting e avançando para Random Forest, AdaBoost, Gradient Boosting, XGBoost, LightGBM e CatBoost.

## 1. Conceito de Ensemble Learning

Ensemble Learning é uma estratégia que combina as previsões de múltiplos modelos com o objetivo de produzir uma previsão final mais robusta e com melhor capacidade de generalização.

A combinação de vários modelos pode reduzir limitações presentes em modelos individuais, principalmente quando existe diversidade entre eles, ou seja, quando não cometem exatamente os mesmos erros.

A forma como os modelos são treinados e como suas previsões são combinadas depende da estratégia de Ensemble utilizada.

## 4. Experimento: Árvore de Decisão × Bagging × Random Forest

Neste experimento, iremos comparar três abordagens:

- uma única Árvore de Decisão;
- Bagging utilizando árvores como modelos-base;
- Random Forest.

O objetivo é observar como a combinação de múltiplas árvores e a introdução de aleatoriedade podem influenciar o desempenho e a capacidade de generalização dos modelos.

In [8]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    random_state=42
)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [4]:
arvore = DecisionTreeClassifier(
    random_state=42
)

arvore.fit(X_train, y_train)

pred_arvore = arvore.predict(X_test)

In [5]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=100,
    random_state=42
)

bagging.fit(X_train, y_train)

pred_bagging = bagging.predict(X_test)

In [6]:
random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest.fit(X_train, y_train)

pred_rf = random_forest.predict(X_test)

In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, matthews_corrcoef

In [10]:
import pandas as pd

resultados = pd.DataFrame({
    "Modelo": [
        "Árvore de Decisão",
        "Bagging",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy_score(y_test, pred_arvore),
        accuracy_score(y_test, pred_bagging),
        accuracy_score(y_test, pred_rf)
    ],
    "Precision": [
        precision_score(y_test, pred_arvore),
        precision_score(y_test, pred_bagging),
        precision_score(y_test, pred_rf)
    ],
    "Recall": [
        recall_score(y_test, pred_arvore),
        recall_score(y_test, pred_bagging),
        recall_score(y_test, pred_rf)
    ],
    "MCC": [
        matthews_corrcoef(y_test, pred_arvore),
        matthews_corrcoef(y_test, pred_bagging),
        matthews_corrcoef(y_test, pred_rf)
    ]
})

resultados.round(3)

,Modelo,Accuracy,Precision,Recall,MCC
0,Árvore de Decisão,0.825,0.810,0.85,0.651
1,Bagging,0.885,0.881,0.89,0.770
2,Random Forest,0.890,0.875,0.91,0.781


### Interpretação

Neste experimento, Bagging e Random Forest apresentaram desempenho superior à Árvore de Decisão individual em todas as métricas avaliadas.

A Random Forest apresentou o maior Recall (0,91), Accuracy (0,89) e MCC (0,781), enquanto o Bagging apresentou Precision ligeiramente superior (0,881 contra 0,875).

Os resultados ilustram como a combinação de múltiplas árvores pode produzir previsões mais robustas do que depender de uma única árvore. Entretanto, a escolha do modelo deve considerar o objetivo do negócio, o custo dos diferentes tipos de erro e uma validação mais robusta em diferentes amostras.

## Comparativo dos principais algoritmos de Boosting

| Algoritmo | Como funciona | Principal diferencial | Pontos fortes | Limitações / cuidados | Quando considerar |
|---|---|---|---|---|---|
| **AdaBoost** | Treina weak learners sequencialmente, aumentando a importância relativa das observações classificadas incorretamente. A combinação final é ponderada. | Foco progressivo nas observações difíceis. | Conceitualmente simples e pode funcionar bem em problemas relativamente limpos. | Pode dar atenção excessiva a ruído e outliers. | Como alternativa de Boosting mais simples, especialmente em bases menores e relativamente limpas. |
| **Gradient Boosting** | Cada novo modelo aprende uma correção relacionada ao gradiente negativo da função de perda. O modelo final é uma soma sequencial das contribuições. | Otimização explícita de uma função de perda por meio de correções sucessivas. | Flexível e é a base conceitual dos Boostings modernos. | Treinamento sequencial e risco de overfitting se a complexidade não for controlada. | Quando queremos um Boosting clássico e controle sobre loss, learning rate e complexidade das árvores. |
| **XGBoost** | Expande a ideia do Gradient Boosting usando informação de primeira e segunda ordem e uma função objetivo regularizada. | Regularização, otimizações computacionais, amostragem de linhas/features e tratamento eficiente de estruturas esparsas e missing values. | Forte desempenho em dados tabulares e muitos mecanismos para controlar o modelo. | Mais hiperparâmetros e maior complexidade de configuração/tuning. | Quando buscamos forte desempenho tabular e queremos bastante controle sobre regularização e complexidade. |
| **LightGBM** | Gradient Boosting otimizado para eficiência, normalmente usando histogramas e crescimento leaf-wise. | Leaf-wise e mecanismos voltados à eficiência, como histogramas; também suporta técnicas como GOSS e EFB. | Muito eficiente e escalável, especialmente em grandes bases. | O crescimento leaf-wise pode favorecer overfitting se a complexidade não for bem controlada. | Bases grandes e cenários em que tempo de treinamento e eficiência computacional são relevantes. |
| **CatBoost** | Boosting com mecanismos específicos para reduzir vieses no tratamento de categóricas e no processo de treinamento. | Ordered Target Statistics, Ordered Boosting e, por padrão, árvores simétricas. | Forte opção para dados tabulares com muitas variáveis categóricas e alta cardinalidade. | Seus mecanismos adicionais podem aumentar o custo de treinamento; sua vantagem depende das características da base. | Especialmente interessante quando existem muitas features categóricas ou categorias de alta cardinalidade. |

### Como pensar na escolha

Não existe um algoritmo universalmente melhor.

A escolha deve considerar:

**problema de negócio → características dos dados → algoritmos candidatos → validação → métrica adequada → desempenho + custo + interpretabilidade → decisão**

Alguns atalhos de raciocínio:

- **AdaBoost:** foco nas observações classificadas incorretamente.
- **Gradient Boosting:** correções guiadas pela função de perda e seu gradiente.
- **XGBoost:** Gradient Boosting com uma "caixa de ferramentas extra" de regularização e otimização.
- **LightGBM:** eficiência computacional e crescimento leaf-wise.
- **CatBoost:** destaque para tratamento de variáveis categóricas e mecanismos ordenados.

> Esses pontos ajudam a selecionar candidatos, mas não substituem a validação empírica.